### Question 
Dataset
You are given a dataset of an online retail store with the following columns: customer_id, age,
annual_income, spending_score, visits_last_month, avg_session_time, purchased (0/1). Goal:
Predict whether a customer will make a purchase in the next visit.
Tasks
1 Load the dataset and check for missing values.
2 Handle missing values appropriately (justify choice).
3 Detect and treat outliers in annual_income and avg_session_time.
4 Perform feature engineering: create income_per_visit = annual_income / (visits_last_month+1).
5 Split dataset into train/test (80/20).
6 Train Logistic Regression and Random Forest.
7 Evaluate using Accuracy, Precision, Recall, F1 Score.
8 Explain which model is better and why (bias vs variance reasoning).
9 Identify top 3 important features.
10 Suggest one business action based on results.
Bonus (If time remains)
Apply standard scaling and check whether Logistic Regression improves.

importing necessary libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Generating synthetic dataset

In [7]:
np.random.seed(42)
n_customers = 1000
customer_id = np.arange(1, n_customers + 1)
age = np.clip(np.random.normal(35, 10, n_customers), 18, 70).astype(int)
annual_income = np.clip(np.random.lognormal(11, 0.5, n_customers) * 1000, 20000, 200000).astype(int)
spending_score = np.clip(np.random.uniform(1, 100, n_customers) + (annual_income / 2000), 1, 100).astype(int)
visits_last_month = np.clip(np.random.poisson(5, n_customers), 0, 50)
avg_session_time = np.clip(np.random.normal(15, 5, n_customers), 1, 60)
linear_combo = (annual_income / 100000) + (visits_last_month / 10) + (avg_session_time / 10) - 1
prob_purchase = 1 / (1 + np.exp(-linear_combo))
purchased = np.random.binomial(1, prob_purchase)

data = pd.DataFrame({
    'customer_id': customer_id,
    'age': age,
    'annual_income': annual_income,
    'spending_score': spending_score,
    'visits_last_month': visits_last_month,
    'avg_session_time': avg_session_time,
    'purchased': purchased
})
print(data.head())
print(f"Dataset shape: {data.shape}")

   customer_id  age  annual_income  spending_score  visits_last_month  \
0            1   39         200000             100                  5   
1            2   33         200000             100                  4   
2            3   41         200000             100                  3   
3            4   50         200000             100                  8   
4            5   32         200000             100                  8   

   avg_session_time  purchased  
0         15.112565          1  
1          3.612187          1  
2         14.381534          1  
3         13.136289          1  
4         17.459545          1  
Dataset shape: (1000, 7)


checking missing values

In [8]:
missing_values = data.isnull().sum()
print("Misiing values per columns:")
print(missing_values)
print(f"Total missing values: {missing_values.sum()}")

Misiing values per columns:
customer_id          0
age                  0
annual_income        0
spending_score       0
visits_last_month    0
avg_session_time     0
purchased            0
dtype: int64
Total missing values: 0


there are no misiing values as it is synthetic dataset.

In [9]:
print("No missing values to handle.")

No missing values to handle.


Task 3: detect and treat outliers (IQR method)

In [10]:
def treat_outliers(col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    data[col] = np.clip(data[col], lower, upper)
treat_outliers("annual_income")
treat_outliers("avg_session_time")

Performing feature engineering

In [11]:
data["income_per_visit"] = data["annual_income"] / (data["visits_last_month"] + 1)

It captures income behavior relative to engagement, it helps model detect customer value per visit.

Train-Test split

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [15]:
X = data.drop(["customer_id", "purchased"], axis=1)
y = data["purchased"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

to ensure class distribution is balanced in both train and test sets.

Training models

Logistic Regression

In [16]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)

Random Forest

In [17]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

Evaluating models

In [18]:
def evaluate_model(y_test, y_pred, model_name):
    print(f"\n{model_name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("precision", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
evaluate_model(y_test, y_pred_log, "LogisticRegression")
evaluate_model(y_test, y_pred_rf, "Random Forest")


LogisticRegression
Accuracy: 0.96
precision 0.96
Recall: 1.0
F1 Score: 0.9795918367346939

Random Forest
Accuracy: 0.96
precision 0.96
Recall: 1.0
F1 Score: 0.9795918367346939


bias vs variance reasoning: Logistic regression: It is a linear model which assumes straight line relationship between features and target, it has high bias and low variance. Because it is simple, it may underfit if data is very complex, but in this case it performed well. Random forest: It is a non-linear model which uses many decision trees, it has low bias, slightly higher variance, but controlled by averaging trees. It can capture feature interaction better, it usually performs better when relationships are complex.

In this case: both models have identical Accuracy, Precision, Recall, and F1 score. This suggects the dataset pattern is not highly complex. Thepurchase behavior can be explained well with a linear decision boundary. Therefore Logistic Regression is preferable here, because it gave same performance as Random forest, simple model, faster to train, easier to interpret, lower risk of overfitting. Although Random Forest is more powerful and can capture complex non-linear relationship, in this dataset both models achieved the same performance. Since Logistic Regression is simpler, more interpretable, and computationally efficient, it is better choice for this problem. This indicates that the relationship between features and purchase behaviour is mostly linear and does not require a highly comple model. 

Top 3 important features

In [19]:
feature_importances = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)
feature_importances.head(3)

avg_session_time     0.589356
age                  0.280897
visits_last_month    0.065477
dtype: float64

(1) avg_session_time(most important): this is the strongest predictor. Customers who spend more time per session are much more likely to purchase. It contributes nearly 59% of the model’s decision-making power. The longer a customer stays on the website, the higher the chance of purchase. (2): age: Age plays a significant role. Certain age groups may have higher purchasing behavior. The business may have a target age segment that buys more frequently. (3)visits_last_month: More visits increase purchase probability. However, compared to session time, it has smaller influence. Engagement frequency matters, but session duration matters more. The model shows that customer engagement behavior (especially session time) is more important than income or spending score in predicting purchases. This suggests that:Keeping customers engaged longer, Improving website experience, Increasing browsing time
can significantly improve conversion rates.

Since avg_session_time is the most important feature, this will increase purchase probability.

Business Action

From the feature importance analysis, the top features influencing purchase are:
(1)avg_session_time (most important),
(2)age, 
(3)visits_last_month
This means customer engagement behavior plays a major role in predicting purchases.
Since avg_session_time is the most important feature, the company should focus on keeping customers engaged longer on the website. The longer customers stay, the higher the probability they will purchase. We can do this by: Improve website design and user experience (UX), Add personalized product recommendations, Show “Customers also bought” suggestions, Offer interactive content (reviews, videos, product demos), Provide limited-time offers while browsing. Target High-Engagement Customers: Customers who: Visit frequently, Spend more time per session, Should receive: Personalized discounts, Email reminders, Loyalty rewards. This increases conversion chances. Age-Based Marketing Strategy: Since age is an important feature: Identify high-purchasing age groups, Run targeted ads for those segments, Customize product recommendations based on age preferences. 

The model shows that customer engagement behavior is more important than income level.
Therefore, instead of targeting only high-income customers, the company should: Focus on increasing engagement, Improve user experience, Retain active visitors. This strategy is likely to increase overall sales and conversion rate.

Standard Scaling

In [20]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model_scaled = LogisticRegression(max_iter=1000)
log_model_scaled.fit(X_train_scaled, y_train)

y_pred_scaled = log_model_scaled.predict(X_test_scaled)

evaluate_model(y_test, y_pred_scaled, "Logistic Regression (Scaled)")


Logistic Regression (Scaled)
Accuracy: 0.96
precision 0.96
Recall: 1.0
F1 Score: 0.9795918367346939


Standard Scaling transforms features so that: Mean = 0, Standard Deviation = 1. It ensures all features are on the same scale. In this case: Performance remained exactly the same. This suggests: The model was already performing optimally. Feature magnitudes were not negatively affecting optimization. The dataset is relatively well-behaved and not highly complex. Although scaling usually improves Logistic Regression performance, in this dataset: The model already achieved high performance. Scaling did not change accuracy or F1 score. This indicates stable feature relationships and good model convergence.However, in real-world datasets with very different feature scales, scaling is strongly recommended. Thus Standard Scaling did not improve performance in this case, but it remains an important preprocessing step for linear models like Logistic Regression to ensure stable and efficient learning.